# DSCI 6883 - Descriptive Analytics & Data Integrity
## Discussion 3.1 Follow-Up: From a Dataset to a Question You Can Answer

**Fall 2026 | Module 3 | Fernando Rubio Garcia**

### Where this came from

Six proposals came in for 3.1, on six very different datasets: baseball, mortgages, laptops, restaurant inspections, AI models, and esports. I am replying to each of you individually about your own dataset. This notebook is about what I noticed **across** them.

The proposals had different data and the same six problems. Each of you ran into one or two, and the problem you have not met yet is probably sitting in your file waiting for week 7. So every section below is built on one of your datasets, and it applies to all of them.

Your replies to each other were good, and some of them are in here. Tyler's reply to Matei asked whether "Pass w/ Conditions" counts as a failure, and his reply to Scott asked which `action_taken` values belong in a denial rate. Those two replies are the same question, and they are Section 1.

If a section is about someone else's dataset, do it anyway. I will point out where it shows up in yours.

### How to use this notebook

1. **Run every cell yourself**, top to bottom, and predict before you run.
2. **Do the `Your Turn` exercises** before opening the answers.
3. **Keep your own dataset open next to this one.** At the end of each section, ask: does my file have this problem?
4. The **assignment is at the end**, and it is on your own dataset. It becomes Sections 1 to 4 of your project notebook.

All the data here is generated in memory, shaped like your real files. Nothing is downloaded or written to disk.

### Outline

| Section | Topic | Built on the dataset from |
|---|---|---|
| 1 | The denominator decides the answer | Matei, Scott |
| 2 | Numbers with words in them | Scott, Jacob, Kara |
| 3 | One cell, several values | Kara |
| 4 | Twenty labels for one thing | Matei, Tyler |
| 5 | Tables that count different things | Tyler |
| 6 | Comparing how fast things grow | Kara |
| 7 | Can your columns answer your question? | Everyone |
| | **Assignment: your project plan** | |

Section 1 is the most important one here, because it changes answers. Sections 2 to 5 are cleaning problems you will write about in your audit. Section 7 is the check to run on your question before you build anything else.

---
## Setup

In [0]:
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas version:", pd.__version__)
rng = np.random.default_rng(6883)

---
# 1. The denominator decides the answer

**From Matei's question:** *"Did food-inspection failure rates change after 2021, and was the change different between restaurants and grocery stores?"*

**From Tyler's reply to Matei:** *"Inspection results often include outcomes such as Pass, Pass w/ Conditions, Fail, and No Entry. Depending on whether you treat 'Pass w/ Conditions' as a pass or a failure, your results could change significantly."*

**From Tyler's reply to Scott:** *"Your denial-rate calculation could vary substantially depending on which categories you include in the denominator."*

Tyler is right twice. "Failure rate" sounds like one number. It is really a fraction, and you choose both the top and the bottom. Here is how much that choice matters.

In [0]:
# Inspection results, shaped like the Chicago file. Before 2021 and after, the mix changes.
results = ["Pass", "Pass w/ Conditions", "Fail", "Out of Business", "No Entry", "Not Ready"]
p_before = [0.55, 0.12, 0.20, 0.06, 0.04, 0.03]
p_after  = [0.40, 0.20, 0.18, 0.12, 0.06, 0.04]

years = np.arange(2016, 2025)
frames = []
for y in years:
    p = p_before if y < 2021 else p_after
    k = 1500 if y != 2020 else 700          # 2020: far fewer inspections
    frames.append(pd.DataFrame({"year": y, "results": rng.choice(results, k, p=p)}))
inspections = pd.concat(frames, ignore_index=True)

print(inspections.shape)
inspections["results"].value_counts()

Now three reasonable definitions of "failure rate," each defensible, each something a real analyst might write:

| Definition | Numerator | Denominator |
|---|---|---|
| A | Fail | every inspection row |
| B | Fail | inspections that actually happened: Pass, Pass w/ Conditions, Fail |
| C | Fail **or** Pass w/ Conditions | inspections that actually happened |

In [0]:
completed = ["Pass", "Pass w/ Conditions", "Fail"]
inspections["period"] = np.where(inspections["year"] < 2021, "2016-2020", "2021-2024")

def rate(df, top, bottom=None):
    base = df if bottom is None else df[df["results"].isin(bottom)]
    return base["results"].isin(top).mean()

summary = pd.DataFrame({
    "A: Fail / all rows":         inspections.groupby("period").apply(lambda d: rate(d, ["Fail"])),
    "B: Fail / completed":        inspections.groupby("period").apply(lambda d: rate(d, ["Fail"], completed)),
    "C: Fail+Cond. / completed":  inspections.groupby("period").apply(
                                      lambda d: rate(d, ["Fail", "Pass w/ Conditions"], completed)),
}).round(3)

summary

Same data. Three answers:

- **A** says failure went **down** after 2021.
- **B** says it barely moved.
- **C** says it went **up**, a lot.

None of these is a coding mistake. A drifts down because more rows after 2021 are "Out of Business" and "No Entry," which are not inspections at all but still sit in the denominator. C goes up because "Pass w/ Conditions" grew.

So which is right? **That depends on your question, and you have to decide it before you look at the results.** If you try all three and report the one that makes a good story, you have manufactured a finding. Write the definition in your notebook first, with the reason, and then compute it.

> **Matei:** B is the usual starting point, since "Out of Business" and "No Entry" are not inspections of how a kitchen is run. Then report C next to it, because whether conditions count as failures is exactly the kind of choice your Limitations section should admit could change the answer.

#### Your Turn 1.1 - Scott's denial rate

HMDA records one row per mortgage application, and `action_taken` is the outcome. The codes are:

| Code | Meaning |
|---|---|
| 1 | Loan originated |
| 2 | Approved, not accepted by the applicant |
| 3 | Denied |
| 4 | Withdrawn by the applicant |
| 5 | File closed for incompleteness |
| 6 | Purchased loan (bought from another lender, not an application decision) |
| 7, 8 | Preapproval request denied / approved but not accepted |

The next cell makes HMDA-shaped data for 2018 to 2025. Compute the denial rate by year **two ways**: denied divided by all rows, and denied divided by applications that got a decision (codes 1, 2, and 3). Do the two versions tell the same story?

In [0]:
hmda_rows = []
for y in range(2018, 2026):
    high_rate_year = y >= 2023
    p = ([0.52, 0.03, 0.11, 0.13, 0.05, 0.14, 0.01, 0.01] if high_rate_year else
         [0.60, 0.03, 0.10, 0.08, 0.03, 0.14, 0.01, 0.01])
    hmda_rows.append(pd.DataFrame({"activity_year": y,
                                   "action_taken": rng.choice(range(1, 9), 3000, p=p)}))
hmda = pd.concat(hmda_rows, ignore_index=True)

# Your work here:

In [0]:
hmda

In [0]:
by_year = hmda.groupby("activity_year")["action_taken"]

all_rows = by_year.apply(lambda s: (s == 3).mean())
decided  = by_year.apply(lambda s: (s[s.isin([1, 2, 3])] == 3).mean())

pd.DataFrame({"denied / all rows": all_rows, "denied / decided (1,2,3)": decided}).round(3)

<details>
<summary><b>Show answer</b></summary>

```python
by_year = hmda.groupby("activity_year")["action_taken"]

all_rows = by_year.apply(lambda s: (s == 3).mean())
decided  = by_year.apply(lambda s: (s[s.isin([1, 2, 3])] == 3).mean())

pd.DataFrame({"denied / all rows": all_rows, "denied / decided (1,2,3)": decided}).round(3)
```

The "decided" version jumps in 2023, from about 14% to about 16.5%. The "all rows" version barely moves, so it would tell Scott that nothing happened. It is watered down by purchased loans (code 6), which are not decisions at all, and it is moved around by withdrawals (code 4), which rose in the high-rate years. Applicants who expect a denial often withdraw first, so withdrawals are partly hidden denials, and that is worth a sentence in your Limitations section, Scott.

The convention in most published HMDA analysis is the second one: **denied / (originated + approved not accepted + denied)**, codes 1, 2, and 3. Whatever you choose, state it and cite the convention.

</details>

---
# 2. Numbers with words in them

**From Scott's post:** *"In the interest rate column the datatype is text not a number. 18.5% of the rows are blank and 6.5% use the word 'Exempt'."*

**From Kara's reply to Jacob:** *"on the RAM it has 8GB and 16GB, etc. You could leave that as a string for categorical grouping... or you could remove the GB and turn it into a integer."*

**From Kara's post:** *"The field 'Training dataset size (total)' is stored as a string, but it appears to be numeric data."*

Three datasets, same problem: a column of numbers with some text mixed in. The fix everyone reaches for is `pd.to_numeric(..., errors="coerce")`, which turns anything that is not a number into `NaN`. It works. It is also how information quietly disappears.

In [0]:
# loans

In [0]:
raw = io.StringIO(
    "loan_id,interest_rate\n"
    "1,3.875\n2,Exempt\n3,\n4,4.25\n5,Exempt\n6,6.99\n7,NA\n8,7.125\n9,5.5\n10,Exempt\n"
)
loans = pd.read_csv(raw)
print(loans.dtypes, "\n")
print(loans["interest_rate"].value_counts(dropna=False))

Already one thing happened that you did not ask for: the text `NA` on row 7 became a real missing value, because `read_csv` treats `NA` as missing by default. The blank on row 3 did too. `Exempt` did not, so the column stayed text.

Now the standard fix:

In [0]:
loans["rate_num"] = pd.to_numeric(loans["interest_rate"], errors="coerce")
print(loans)
print("\nmissing rate_num:", loans["rate_num"].isna().sum())

It worked, and now "the lender is exempt from reporting this" and "this was blank" are **the same `NaN`**. They are not the same thing. Exempt applications come from smaller lenders, and small lenders are not a random slice of the market. If exempt lenders deny at a different rate, your analysis needs to know which rows they are.

Two habits fix this.

**Habit 1: before converting, see exactly what you are about to throw away.**

In [0]:
def coerce_report(series):
    'Convert to numbers, and print every value that was text and became NaN.'
    converted = pd.to_numeric(series, errors="coerce")
    lost = series[converted.isna() & series.notna()]
    print(f"{len(lost)} values were not numbers and became NaN:")
    print(lost.value_counts().to_string())
    return converted

loans["rate_num"] = coerce_report(loans["interest_rate"])

**Habit 2: if the text means something, keep it in its own column before converting.**

In [0]:
loans["rate_status"] = np.select(
    [loans["interest_rate"] == "Exempt", loans["rate_num"].isna()],
    ["exempt", "blank"],
    default="reported",
)
loans[["interest_rate", "rate_num", "rate_status"]]

Now `rate_num` is a real number column you can average, and `rate_status` keeps the reason for every gap. You can report "18.5% blank, 6.5% exempt" in your audit, and you can check later whether the exempt rows behave differently.

### 2.1 Units glued to the number

Jacob's laptop file has the other version of this problem: the text is a unit, stuck to every value. That is easier, because the unit carries no information once you know what it is. Strip it and convert.

In [0]:
laptops = pd.DataFrame({
    "Ram":    ["8GB", "16GB", "4GB", "8GB", "32GB", "8GB", "12GB"],
    "Weight": ["1.37kg", "2.1kg", "?", "1.86kg", "4.42kgs", "1.3kg", "2kg"],
})

laptops["ram_gb"] = pd.to_numeric(laptops["Ram"].str.replace("GB", "", regex=False))

# 'kgs?' means 'kg' followed by an optional 's'. Then report what could not convert.
laptops["weight_kg"] = coerce_report(laptops["Weight"].str.replace(r"kgs?$", "", regex=True))

laptops

Two details:

- `coerce_report` caught the `?`. That is a disguised missing value, the kind from Section 5 of the 2.2 follow-up. It would have become `NaN` either way, but now you know it happened and can count it in your audit.
- Kara's point about RAM is a good one and worth a decision: `ram_gb` as a number lets you compute an average; RAM as a category (4, 8, 12, 16, 32) makes cleaner groups for a chart. You can have both columns. That is not indecision; they answer different questions.

> **A note on formatting vs data.** Jacob mentioned that price has four decimals and no dollar sign. That is display, not data. Keep the column as a plain float and format it when you show it, for example `f"${price:,.2f}"`. Storing `"$1,339.69"` as text would undo all of Section 2. Do check the **currency**, though: a price column with no symbol is a question, not an answer.

---
# 3. One cell, several values

**From Kara's post:** *"The field 'Organization categorization' has some rows with multiple entries, i.e. 'Industry,Academia,Industry,Academia,Academia.' The field 'Country (of organization)' has a similar issue. i.e. there are 79 records that say 'United States of America,United States of America.'"*

**From Matei's reply to Kara:** *"The repeated country and organization entries will also need to be standardized before grouping the data."*

Kara found something that shows up in any dataset where one thing can belong to several groups: a paper with several authors, a movie with several genres, a model built by several organizations. The cell holds a list, written as text.

The cleaning is mechanical. The **counting** is a decision, and it changes your answer.

In [0]:
models = pd.DataFrame({
    "model":   ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"],
    "org_cat": ["Industry", "Academia", "Industry,Academia",
                "Industry,Academia,Industry,Academia,Academia",
                "Industry", "Academia,Industry", "Industry,Industry", None],
    "country": ["United States of America", "China",
                "United States of America,United States of America",
                "United Kingdom,United States of America",
                "United States of America", "China,United States of America",
                "United States of America", "France"],
})
models

### 3.1 The mechanical part: split, trim, de-duplicate, sort

`"Industry,Academia"` and `"Academia,Industry"` are the same thing written in a different order, and `"Industry,Industry"` is just `"Industry"`. One function fixes all of that:

In [0]:
def clean_multi(value, sep=","):
    'Turn "b, a,b" into "a + b": split, trim, drop repeats, sort.'
    if pd.isna(value):
        return value
    parts = sorted({p.strip() for p in str(value).split(sep) if p.strip()})
    return " + ".join(parts)

models["org_clean"] = models["org_cat"].map(clean_multi)
models["country_clean"] = models["country"].map(clean_multi)
models[["model", "org_cat", "org_clean", "country_clean"]]

Kara's `"Industry,Academia,Industry,Academia,Academia"` is now `"Academia + Industry"`, the same as M3 and M6. And the 79 double-USA rows collapse to plain `"United States of America"`.

### 3.2 The decision: how do you count a model that belongs to two groups?

Kara wants to compare US models to others. What do you do with M4 (UK and US) and M6 (China and US)? Three policies:

In [0]:
# Policy 1: "combination as its own group". Every model counted once.
policy_1 = models["country_clean"].value_counts()

# Policy 2: "counts for each". A two-country model counts once for each country.
exploded = models.assign(country_one=models["country_clean"].str.split(" + ", regex=False)).explode("country_one")
policy_2 = exploded["country_one"].value_counts()

# Policy 3: "fractional". A two-country model counts one half for each.
exploded["weight"] = 1 / exploded.groupby("model")["model"].transform("size")
policy_3 = exploded.groupby("country_one")["weight"].sum()

print("Policy 1, combinations as groups:\n", policy_1.to_string(), "\n  total:", policy_1.sum(), "\n")
print("Policy 2, counts for each:\n", policy_2.to_string(), "\n  total:", policy_2.sum(), "\n")
print("Policy 3, fractional:\n", policy_3.round(2).to_string(), "\n  total:", policy_3.sum())

The US has 4, 6, or 5 models depending on the policy. Policy 2's total is 10 for 8 models, so any percentage you compute from it adds up to more than 100%.

| Policy | Good for | Watch out for |
|---|---|---|
| 1. Combination as a group | "Are collaborations different?" | Many tiny groups |
| 2. Counts for each | "How many models did the US take part in?" | Totals exceed the number of models; no shares |
| 3. Fractional | "What share of models came from the US?" | Harder to explain to a non-technical reader |

For Kara's question, comparing growth across countries, policy 2 or 3 works, as long as you say which one. `explode` is the tool you will reach for either way: it turns one row with a list into one row per list item. Any dataset with a "genres", "tags", "authors", or "categories" column needs it.

---
# 4. Twenty labels for one thing

**From Matei's post:** *"The Facility Type column contains inconsistent descriptions for similar businesses. For example, restaurant-related facilities may be recorded using different labels or combinations of words."*

**From Tyler's post:** *"because the database spans more than a century of baseball history, there may be inconsistencies in team names."*

When a column is typed in by people, the same thing gets written many ways. Your question compares groups ("restaurants vs grocery stores," "eras," "teams"), so you have to build the groups before you can compare them. Here is a method that works on any column like this.

In [0]:
facility_counts = {
    "Restaurant": 900, "RESTAURANT": 60, "restaurant ": 25, "Restaurant/Bar": 40,
    "Restaurant/Bakery": 15, "Grocery Store": 300, "GROCERY STORE": 30, "grocery store": 12,
    "Grocery & Restaurant": 45, "Gas station/Grocery": 20, "Grocery(Sushi prep)": 6,
    "Bakery": 80, "Daycare (2 - 6 Years)": 60, "CHILDRENS SERVICES FACILITY": 25,
    "Liquor store": 18, "Mobile Food Truck": 14,
}
facility = pd.Series(np.repeat(list(facility_counts), list(facility_counts.values())), name="facility_type")

# Give each inspection a Fail/Pass, with grocery-type places failing a bit more often.
fail_p = np.where(facility.str.lower().str.contains("grocer"), 0.26, 0.20)
food = pd.DataFrame({"facility_type": facility, "failed": rng.random(len(facility)) < fail_p})

print(food["facility_type"].nunique(), "distinct labels")
food["facility_type"].value_counts()

### Step 1: normalize the text, then count again

Case, stray spaces, and double spaces are not real differences. Remove them first, so what is left is the real variety.

In [0]:
food["ft_norm"] = (food["facility_type"]
                   .str.strip()
                   .str.lower()
                   .str.replace(r"\s+", " ", regex=True))

print(food["facility_type"].nunique(), "labels before ->", food["ft_norm"].nunique(), "after normalizing")
food["ft_norm"].value_counts()

### Step 2: write rules, in order, and check what they caught

Each rule is a pattern and the group it assigns. **Order matters**: the first matching rule wins. `np.select` does exactly this.

In [0]:
rules = [
    ("grocery",    food["ft_norm"].str.contains("grocer")),
    ("restaurant", food["ft_norm"].str.contains("restaurant|bakery|food truck")),
    ("childcare",  food["ft_norm"].str.contains("daycare|children")),
]
food["ft_group"] = np.select([r[1] for r in rules], [r[0] for r in rules], default="other")

# The check: which original labels went into each group?
food.groupby(["ft_group", "ft_norm"]).size()

Always print that table. It is the only way to see a rule catching something you did not mean it to. Here, look at `grocery & restaurant`: because the grocery rule comes first, it went into **grocery**. Flip the order of the two rules and it goes into restaurant. Neither is wrong. It is a judgment call, and **a judgment call is something you test.**

### Step 3: test whether the judgment call changes your answer

In [0]:
ambiguous = food["ft_norm"].isin(["grocery & restaurant", "gas station/grocery"])

version_a = food.copy()                                            # ambiguous ones count as grocery
version_b = food.copy()
version_b.loc[ambiguous, "ft_group"] = "restaurant"                 # ambiguous ones count as restaurant

compare = pd.DataFrame({
    "ambiguous -> grocery":    version_a.groupby("ft_group")["failed"].mean(),
    "ambiguous -> restaurant": version_b.groupby("ft_group")["failed"].mean(),
}).loc[["grocery", "restaurant"]].round(3)
compare

If the gap between grocery and restaurant is about the same both ways, your conclusion does not depend on the judgment call, and you can say so in one sentence. If the gap changes a lot, that is a finding about your data, and it belongs in your Limitations section.

This is called a **sensitivity check**, and it is the most useful habit in this notebook for your project. Any time you make a cleaning decision you are unsure about, run the analysis both ways.

> **Tyler:** the Lahman `Teams` table already solves your version of this. Besides `teamID`, it has a `franchID` column that links a franchise across moves and renames. When a dataset already has a clean grouping column, use it instead of building your own, and say that you did.

---
# 5. Tables that count different things

**From Tyler's post:** *"player information is spread across multiple tables that must be merged correctly using player IDs. This creates opportunities for duplicate records or missing matches."*

Tyler already saw the risk. The Lahman database makes it concrete, because its tables count different things:

- In `Batting`, one row is **one player, one season, one team**. A player traded mid-season has two rows that year.
- In `HallOfFame`, one row is **one player on one year's ballot**. A player who was on the ballot for eight years has eight rows.

That "what is one row?" is called the table's **grain**. Merging two tables with different grains, without thinking about it, multiplies rows.

In [0]:
# Batting-shaped data: player-season-stint rows.
players = [f"p{i:03d}" for i in range(200)]
bat_rows = []
for pid in players:
    start = int(rng.integers(1960, 2008))
    seasons = int(rng.integers(2, 21))
    talent = rng.gamma(2.0, 0.5)
    for yr in range(start, min(start + seasons, 2016)):
        stints = 2 if rng.random() < 0.08 else 1
        for stint in range(1, stints + 1):
            bat_rows.append({"playerID": pid, "yearID": yr, "stint": stint,
                             "H": int(rng.poisson(90 * talent / stints)),
                             "HR": int(rng.poisson(12 * talent / stints))})
batting = pd.DataFrame(bat_rows)

batting.head(5)

In [0]:
career_h[career_h > career_h.quantile(0.80)]

In [0]:


# HallOfFame-shaped data: one row per player per ballot year.
career_h = batting.groupby("playerID")["H"].sum()
candidates = career_h[career_h > career_h.quantile(0.80)].index
hof_rows = []
for pid in candidates:
    final = batting.loc[batting["playerID"] == pid, "yearID"].max()
    n_ballots = int(rng.integers(1, 9))
    inducted_at = n_ballots if career_h[pid] > career_h.quantile(0.93) else None
    for b in range(1, n_ballots + 1):
        hof_rows.append({"playerID": pid, "yearid": final + 5 + b,
                         "inducted": "Y" if b == inducted_at else "N", "category": "Player"})
hall = pd.DataFrame(hof_rows)

print("batting:", batting.shape, "  hall:", hall.shape)

In [0]:
hall

Note the column names: `yearID` in batting, `yearid` in the Hall of Fame table. That is how it is spelled in the Lahman files I have seen. Check yours, and run `to_snake` from 2.2 - 2.4 Questions on every table before you merge.

### 5.1 The naive merge

In [0]:
naive = batting.merge(hall, on="playerID", how="inner")
print("batting rows for players who were ever on a ballot:",
      batting["playerID"].isin(hall["playerID"]).sum())
print("rows after the merge:", len(naive))

Every season row got copied once per ballot year. A player with 15 seasons and 8 ballots now has 120 rows, and his career hits are counted eight times. Nothing errored. (The `validate="many_to_one"` check from 2.2 - 2.4 Questions would have caught this.)

### 5.2 The fix: bring both tables to the grain of your question

Tyler's question is about **players**, so both tables need one row per player *before* the merge.

In [0]:
# Batting -> one row per player (career totals).
careers = (batting.groupby("playerID")
           .agg(first_year=("yearID", "min"), last_year=("yearID", "max"),
                seasons=("yearID", "nunique"), H=("H", "sum"), HR=("HR", "sum"))
           .reset_index())
careers.head(5)

In [0]:


# HallOfFame -> one row per player: were they ever inducted as a player?
hof_player = (hall[hall["category"] == "Player"]
              .groupby("playerID")["inducted"]
              .apply(lambda s: (s == "Y").any())
              .rename("inducted")
              .reset_index())

hof_player.head(5)

In [0]:


merged = careers.merge(hof_player, on="playerID", how="left", validate="one_to_one")
merged["inducted"] = merged["inducted"].fillna(False).astype(bool)

print("careers:", len(careers), "  merged:", len(merged))
print(merged["inducted"].value_counts())

One row per player, and the row count did not change. `nunique` on `yearID` counts seasons correctly even for traded players, where a plain row count would count one season twice.

### 5.3 The comparison group is a decision too

Tyler's question compares inducted players with everyone else. But who is "everyone else"? A player who retired two years before the data ends **cannot** be in the Hall of Fame yet: players have to be retired for five years before they are eligible, and they need ten seasons. Counting them as "not inducted" is counting them as failures when they have not been judged.

In [0]:
data_ends = batting["yearID"].max()

populations = {
    "everyone":                         merged,
    "10+ seasons":                      merged[merged["seasons"] >= 10],
    "10+ seasons, retired 5+ years":    merged[(merged["seasons"] >= 10) &
                                               (merged["last_year"] <= data_ends - 5)],
}
pd.DataFrame([{"population": name, "players": len(df), "inducted": int(df["inducted"].sum()),
               "induction rate": round(df["inducted"].mean(), 3)}
              for name, df in populations.items()]).set_index("population")

That is Section 1 again, in a different dataset: **the denominator decides the answer.** "Which stats are associated with induction?" has to be asked among players who could have been inducted.

> **Tyler, two more for your list.** Pitchers are judged on the `Pitching` table, not `Batting`, so decide early whether your question is about position players only. And players who were eligible but linked to steroids were kept out by voters despite Hall of Fame numbers. They are real data, not errors, and they may be the most interesting rows you have.

---
# 6. Comparing how fast things grow

**From Kara's question:** *"are there differences in the rate of change between organizations (i.e. Open AI vs Google), country (i.e. US vs Others), and domain (i.e. Language vs. Visual)?"*

**From Matei's reply:** *"You could narrow it to whether AI training compute has grown differently for U.S. models compared with models from other countries since 2010."*

Matei's narrowing is good, and I will use it. "Rate of change" has a specific meaning when a quantity grows by multiplying, and training compute grows by multiplying, several times over every year. This section shows how to measure that, and one trap in Kara's data that could bias it.

In [0]:
# Models 2012-2024 with training compute in FLOP. log10(compute) grows along a straight line.
def make_group(name, n, start, slope, base):
    year = rng.uniform(start, 2024.9, n)
    log_c = base + slope * (year - 2012) + rng.normal(0, 0.6, n)
    return pd.DataFrame({"group": name, "year": year, "log10_compute": log_c})

ai = pd.concat([
    make_group("United States", 260, 2012, 0.55, 18.0),
    make_group("Other",         110, 2014, 0.50, 17.8),
], ignore_index=True)
ai["compute"] = 10 ** ai["log10_compute"]

print(ai.groupby("group").size())

In [0]:
ai

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for g, d in ai.groupby("group"):
    axes[0].scatter(d["year"], d["compute"], s=8, label=g, alpha=0.6)
    axes[1].scatter(d["year"], d["compute"], s=8, label=g, alpha=0.6)
axes[0].set_title("Linear axis: everything before 2022 is a flat line")
axes[1].set_yscale("log")
axes[1].set_title("Log axis: the growth is a straight line")
for ax in axes:
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Training compute (FLOP)")
    ax.legend()
plt.tight_layout()
plt.show()

On the left, the biggest recent models squash everything else flat. On the right, each group is a straight line, and **the slope of that line is the growth rate**. That is the chart Epoch uses too.

A straight line on a log axis means "multiplies by the same factor every year," so the slope of `log10(compute)` against `year` translates directly into plain English:

In [0]:
def growth(d):
    slope, _ = np.polyfit(d["year"], d["log10_compute"], 1)
    return pd.Series({
        "models":            len(d),
        "log10 per year":    round(slope, 3),
        "x per year":        round(10 ** slope, 2),
        "doubling (months)": round(12 * np.log10(2) / slope, 1),
    })

ai.groupby("group").apply(growth).astype({"models": int})

"Compute grew about 3.5x per year, doubling every 6 to 7 months" is a sentence a manager can use, and it is the number you compare between groups.

### 6.1 The trap: which models report their compute?

Kara wrote, *"There is a lot of missing data."* Here is why that matters for a growth rate. Suppose smaller models are less likely to report compute, which is plausible, since big labs publicize big numbers. Let us hide values that way and fit again:

In [0]:
# The smaller the model, the more likely its compute is missing.
size_rank = ai.groupby(ai["year"].round())["log10_compute"].rank(pct=True)
ai["reported"] = rng.random(len(ai)) < (0.25 + 0.7 * size_rank)

print(f"reported: {ai['reported'].mean():.0%} of models\n")
print("all models (the truth we never see):")
print(ai.groupby("group").apply(growth).astype({"models": int})[["models", "x per year"]], "\n")
print("only models that reported compute (what you would analyze):")
print(ai[ai["reported"]].groupby("group").apply(growth).astype({"models": int})[["models", "x per year"]])

A third of the models dropped out, and the growth rates barely moved. That is the lucky case, and it happened because I made the missingness the same in every year: small models were equally likely to go missing in 2013 and in 2023.

Real reporting habits are not that tidy. If early models reported compute mostly when they were large, and recent ones report it more often, then the early points are biased upward and the recent ones are not, and the slope comes out **too flat**. The same growth rate would look slower than it was. And if the pattern differs between the US and everyone else, the comparison between them moves too.

You cannot check this against the truth in real data. What you can do:

1. Report the share missing **by year and by group**, not only overall.
2. Say which direction the bias would push your answer.
3. Watch the group sizes. A slope fitted on 12 models is a guess. If "Google vs OpenAI" gives you 30 and 15 models with compute, compare countries instead.

---
# 7. Can your columns answer your question?

**From my 3.1 prompt:** *"The test: could the answer come out either way?"*

That test catches vague questions. There is a second test that catches questions your data cannot answer, and it is just as important: **point to the columns.** For each part of your question, name the column that answers it, and how.

| Part | What to write down |
|---|---|
| **Unit** | What is one row? Is it the thing your question is about, or do you have to aggregate first (Section 5)? |
| **Outcome** | Which column holds the answer? If it is a rate, what are the numerator and the denominator (Section 1)? |
| **Groups** | Which column defines the groups you compare? Does it need building first (Sections 3 and 4)? |
| **Time** | Which date column splits before and after? Did anything else change at that date? |
| **Either way?** | What result would surprise you? |

If you cannot fill a row, the question needs to change, or you need another column.

### 7.1 The six proposals, through that test

This is not a grade. It is what I would check first, and each of you will see more detail in my reply to your post.

| | Unit | Outcome | Groups | Time | First thing to check |
|---|---|---|---|---|---|
| **Tyler**, Hall of Fame | Player career (aggregate `Batting` first) | Inducted, among *eligible* players | Era: by debut or by final season? | `yearID` | Batting or pitching? And "most strongly associated" needs a method: compare stat distributions for inducted vs not, era by era. |
| **Scott**, HMDA denials | Application | Denied / decided (codes 1-3) | Tract income vs MSA median | `activity_year` | Is `interest_rate` blank for denied applications? If so, the rate has to come from outside the file (market rate by year), which gives you 8 data points for that half. The tract comparison is the stronger half. |
| **Jacob**, laptops | Laptop listing | ? | Specs | ? | "Good" is not a column. `Price` is. See below. |
| **Matei**, inspections | Inspection | Fail / completed (Section 1) | Facility group (Section 4) | Inspection date | Some rows are re-inspections of a place that just failed. Check `Inspection Type` and consider routine inspections only. Also check whether the rules changed during your period; the portal's description notes a food-code change in 2018. |
| **Kara**, AI models | Model | `log10` training compute, slope per year (Section 6) | Country (Section 3) | Publication date | How many models *with compute reported* are in each group, by year? |
| **Thomas**, esports | ? | `Match_Outcome`, win rate | Reaction time and biometric levels | none | How was this data made? The Kaggle page says it is synthetic. See 7.3. |

### 7.2 Sharpening a question: Jacob's, as the worked example

Jacob's question was *"What makes a good laptop: are there spec standards that need to be met for a laptop to be produced, and if so, what are they?"*

Jacob, the curiosity behind this is exactly right, and your reason for picking it (building your own machines, and wondering whether companies work the same way) is the best "why" in the thread. The problem is the columns. There is no column for "good," and a dataset of laptops that exist cannot show what stops a laptop from being produced, because the ones that were not produced are not in it.

What the file does have is `Price`, and a price is a column that can answer questions. Some versions that pass both tests:

1. *"Which spec explains the most price difference: RAM, CPU family, GPU, or brand?"*
2. *"Is there a brand premium? For laptops with the same RAM and CPU family, how much more do the premium brands charge?"*
3. *"Is there a minimum spec floor? Below what RAM and storage do laptops essentially stop existing in this market, and does that floor differ by laptop type?"*

The third one is closest to your original "standards" idea, and it can be answered with `value_counts` and a few `groupby`s. Any of the three could come out either way.

**The general move:** find the column closest to the word you cannot measure ("good," "successful," "better"), and ask about that column instead.

### 7.3 Where the data came from: Thomas's, as the worked example

Thomas's question was *"Do biometric measurements and reaction times have a significant relationship with winning esports matches, and are some factors stronger predictors of success than others?"*

Thomas, the question passes the first test: it could come out either way, and you said so. You also read the dataset page closely enough to find the author's note about the missing values, which most people skip. That same page has the sentence that rules this dataset out:

> *"This is a **synthetic dataset** ... It was generated using logical, mathematical correlations based on current gaming science (e.g., lower sleep increases reaction time...)."*

Synthetic means nobody played these 250,000 matches. The author wrote formulas and a program drew the rows from them. That causes two problems for this project:

1. **The answer is already decided.** If you find that faster reaction times go with more wins, you have found the formula the author typed in. Your question is about gamers, and the data can only tell you about the author's assumptions.
2. **There is nothing to find.** You wrote *"At first glance, I am not seeing anything wrong."* That is not you missing something. The only problems in this file are the ones the author put there on purpose (`NaN` in `HRV_Score` and `Screen_Brightness`), and the page tells you where they are. This course is about the problems nobody warned you about: the disguised missing values, the labels typed twenty ways, the rows that count different things. A generated file does not have those.

So I am asking you to change the dataset, and keep the topic. Real esports data exists: professional match statistics, for example, are published by fan and analytics sites for games like League of Legends, Dota 2, and Counter-Strike. Real biometric data from real players is almost never public, so the biometric half of your question will probably have to go. What you can keep is the idea behind it, *what separates winning from losing*, asked of columns that recorded real matches. For example: *"Do teams that take the first objective win more often, and has that edge changed across seasons?"*

Post your new dataset as a reply to your 3.1 post, with the link and the first few rows, and I will approve it or redirect it quickly. Your project plan is due soon, so pick something this week.

**The general move:** before you trust a dataset, find out how it was made. If the description says *synthetic*, *simulated*, or *generated*, it can teach you pandas, but it cannot tell you anything about the world.

#### Your Turn 7.1 - Fill the table for a question that is not yours

Pick a classmate's question from 7.1 and fill in all five rows of the table from the top of Section 7 for it, in more detail than I did. Then do the same for your own. It is much easier to see the gaps in someone else's question first.

---
# Where this leaves us

1. **A rate is a fraction, and you choose the top and the bottom.** Decide before you look, write down why, and report the alternative when it is close.
2. **`errors="coerce"` throws information away.** Print what it is about to discard, and keep meaningful text like "Exempt" in its own column.
3. **A cell with a list in it needs a counting policy**, not just cleaning. Say which policy you used.
4. **When you build groups by rules, print what each rule caught, and test the judgment calls both ways.**
5. **Get every table to the grain of your question before merging**, and choose the comparison group as carefully as the outcome.
6. **For things that multiply, compare slopes on a log scale**, and look at who is missing before you trust them.
7. **Point to the column.** If no column answers part of your question, change the question now, not in week 7.
8. **Find out how your data was made.** Synthetic data can teach you pandas, but it cannot tell you anything about the world.

---
# Assignment: Your Project Plan

This is the first real piece of your project notebook. Start the notebook you will submit for 3.2, and build its first four sections now, on your own dataset. Use the section headings from the project instructions. When you finish, you should have a **clean DataFrame** that next week's analysis and charts can start from.

**1. The question.** Your question, revised after my reply and your classmates' replies. Below it, the table from Section 7, all five rows filled in for your question. For **Outcome**, write the exact numerator and denominator (or the exact column and statistic).

**2. The data.** Load your file, with every loading argument you needed visible in the cell. Then:
- apply `to_snake` to the column names (from 2.2 - 2.4 Questions),
- run `first_look` (from the 2.2 follow-up),
- in markdown, say what one row of your file represents,
- compute your outcome **once**, on the data as it is, before any cleaning. One number, or one small table by group. This is not your result. It is proof that your columns can produce one. If you cannot compute it, say what is in the way; that is the most useful thing you can learn this week.

**3. Integrity audit.** Assignment 1, on the columns your question uses. Counts and rates, not impressions:
- missing values by column, both the obvious and the disguised kind (`?`, `"NA"`, `0`, `"Exempt"`, blank text),
- duplicates: full duplicate rows, and repeats in the column that should identify a row,
- columns whose dtype is wrong,
- values that are impossible, not just unusual,
- any problem from this notebook that your file has: a denominator choice, text in a number column, a multi-value cell, labels to group, tables at different grains, missing values that are not random.

**4. Cleaning.** Fix every problem from Section 3, or leave it and say why. For each one:
- the code that fixes it, and a check that proves it worked (a count before and after),
- a row in a decision log (Section 6 of 2.2 - 2.4 Questions): what you did, and what it cost.

At least one decision should come from a section of this notebook. If a decision could reasonably go either way, run your outcome both ways (the sensitivity check in Section 4).

End the section by computing your outcome again on the clean data, next to the number from Section 2, with one sentence on what changed and which decision changed it.

**Before you submit:** *Kernel > Restart Kernel and Run All Cells.* Submit the `.ipynb`, and your data file if it is under 25 MB (otherwise the download link).